In [1]:
import pandas as pd
import glob
import os

# 1. Setup paths
input_path = '../data_inputs/*.csv' # Look for all CSVs in that folder
files = glob.glob(input_path)

# 2. This list will store any errors we find
all_errors = []

print(f"Starting analysis on {len(files)} broker files...\n")

for file in files:
    # Get the broker name from the filename
    broker_name = os.path.basename(file).split('_')[0]
    df = pd.read_csv(file)
    
    # --- CHECK 1: Missing Prices (Nulls) ---
    missing_prices = df[df['Price'].isnull()]
    if not missing_prices.empty:
        for _, row in missing_prices.iterrows():
            all_errors.append([broker_name, row['TradeID'], 'Missing Price', 'CRITICAL'])

    # --- CHECK 2: Price Anomalies (Outliers) ---
    # In finance, a price of 0 or negative is usually a system error
    anomalies = df[(df['Price'] > 5000) | (df['Price'] <= 0)]
    if not anomalies.empty:
        for _, row in anomalies.iterrows():
            all_errors.append([broker_name, row['TradeID'], f"Price Anomaly: {row['Price']}", 'HIGH'])

    # --- CHECK 3: Duplicate Trade IDs ---
    duplicates = df[df.duplicated('TradeID', keep=False)]
    if not duplicates.empty:
        # We only want to log the ID once
        unique_dup_ids = duplicates['TradeID'].unique()
        for tid in unique_dup_ids:
            all_errors.append([broker_name, tid, 'Duplicate Trade ID', 'MEDIUM'])

# 3. Convert errors to a Summary Table
error_df = pd.DataFrame(all_errors, columns=['Broker', 'TradeID', 'Issue_Type', 'Severity'])

print(f"Analysis Complete! Found {len(error_df)} total issues.")
error_df.head(10) # Show the first 10 errors found

Starting analysis on 5 broker files...

Analysis Complete! Found 164 total issues.


,Broker,TradeID,Issue_Type,Severity
0,Apex,TRD-APE-10039,Missing Price,CRITICAL
1,Apex,TRD-APE-10046,Missing Price,CRITICAL
2,Apex,TRD-APE-10249,Missing Price,CRITICAL
3,Apex,TRD-APE-10329,Missing Price,CRITICAL
4,Apex,TRD-APE-10423,Missing Price,CRITICAL
5,Apex,TRD-APE-10433,Missing Price,CRITICAL
6,Apex,TRD-APE-10482,Missing Price,CRITICAL
7,Apex,TRD-APE-10491,Missing Price,CRITICAL
8,Apex,TRD-APE-10505,Missing Price,CRITICAL
9,Apex,TRD-APE-10563,Missing Price,CRITICAL


In [2]:
# 1. Total issues by Broker
print("=== TOTAL ISSUES BY BROKER ===")
broker_summary = error_df.groupby('Broker').size().reset_index(name='Total_Errors')
print(broker_summary)
print("\n" + "="*30 + "\n")

# 2. Total issues by Severity
print("=== BREAKDOWN BY SEVERITY ===")
severity_summary = error_df.groupby('Severity').size().reset_index(name='Count')
print(severity_summary)
print("\n" + "="*30 + "\n")

# 3. Pivot Table: See exactly what issues each broker is struggling with
print("=== BROKER VS ISSUE TYPE MATRIX ===")
matrix_summary = pd.crosstab(error_df['Broker'], error_df['Issue_Type'])
print(matrix_summary)

=== TOTAL ISSUES BY BROKER ===
     Broker  Total_Errors
0      Apex            39
1    Beacon            39
2   Horizon            29
3    Summit            23
4  Vanguard            34


=== BREAKDOWN BY SEVERITY ===
   Severity  Count
0  CRITICAL    101
1      HIGH     53
2    MEDIUM     10


=== BROKER VS ISSUE TYPE MATRIX ===
Issue_Type  Duplicate Trade ID  Missing Price  Price Anomaly: -50.0  \
Broker                                                                
Apex                         5             19                    11   
Beacon                       5             21                     8   
Horizon                      0             23                     4   
Summit                       0             13                     3   
Vanguard                     0             25                     5   

Issue_Type  Price Anomaly: 999999.99  
Broker                                
Apex                               4  
Beacon                             5  
Horizon      

In [3]:
# Create a folder for our cleaned/processed outputs if it doesn't exist
os.makedirs('../data_outputs', exist_ok=True)

# Save the detailed error log
error_df.to_csv('../data_outputs/detailed_error_log.csv', index=False)

# Save our aggregated metrics for Power BI/SQL staging
broker_summary.to_csv('../data_outputs/broker_error_summary.csv', index=False)
matrix_summary.to_csv('../data_outputs/broker_issue_matrix.csv')

print("Files successfully exported to the 'data_outputs' folder! 🚀")

Files successfully exported to the 'data_outputs' folder! 🚀
